# 01 — Collect a Training Dataset

Three notebooks build a complete MOUSE agent end-to-end:

| Notebook | What it does |
|---|---|
| **01 — Collect** (this one) | Run Procedural Frozen Lake with a curriculum policy and save transitions to the Hub |
| **02 — Train** | Load those transitions and train a model offline |
| **03 — Inference** | Load the trained model and watch it solve the maze |

**What this notebook produces:** a Hugging Face dataset with four named configs (`proc_frozenlake_0` … `proc_frozenlake_3`), one per environment slot, each containing 1,500 transitions ready for the training notebook.

### Why a curriculum?

A fully random policy almost never reaches the goal, so a purely random dataset contains almost no reward signal for the model to learn from. We blend random actions with oracle actions — the provably-optimal move computed by value iteration — using a **linear ramp**: `oracle_prob = t / (T − 1)`. Early steps explore broadly; later steps follow the expert. This gives the model examples of both recovery and successful goal-reaching.

In [24]:
import torch

from mouse_envs import EnvConfig, make_env
from mouse_core.data import Datastore, push_stores_to_hub

DATASET_ID  = "mouse-example-dataset"  # created under your authenticated HF user
TOTAL_STEPS = 1500                     # steps per slot; oracle_prob ramps 0 → 1 linearly

## Build Environment

`EnvConfig` describes the environment you want to collect from. `make_env` instantiates it and handles all the Gymnasium wiring.

Key parameters for this run:

- **`num_envs=4`** — four independent slots run in parallel, each with its own 4×4 map generated from `seed+0` … `seed+3`. A single pass collects diverse layouts automatically.
- **`episodes_per_task=10`** — each slot runs ten episodes before the map resets to a new procedurally-generated layout. The model must adapt to both within-task continuity and between-task layout changes.
- **`q_star_source={"provider": "metadata_q_star"}`** — runs value iteration on every reset and attaches the exact optimal Q-values to every step output as `info_metadata_q_star`. These are used as training labels in notebook 02; the model never sees them at inference time.

In [25]:
config = EnvConfig(
    "Procedural-FrozenLake-v1",
    name="proc_frozenlake",
    seed=0,
    num_envs=4,
    episodes_per_task=10,
    kwargs={
        "is_slippery": False,
        "min_width": 4, "max_width": 4,
        "min_height": 4, "max_height": 4,
        "step_penalty": -0.01,
    },
    q_star_source={"provider": "env_q_star"},
)
env = make_env(config)
print("Environments:", env.names)

Environments: ('proc_frozenlake_0', 'proc_frozenlake_1', 'proc_frozenlake_2', 'proc_frozenlake_3')


## Create Datastores

A `Datastore` is a sequential container for step data backed by a Hugging Face Arrow dataset. Each row is a plain Python dict — no fixed schema required. Whatever fields your environment returns, you can store them as-is.

The `name` you give a store becomes its config identifier on the Hub. `load_stores_from_hub` in the training notebook uses that name to fetch the right store back by ID.

In [26]:
stores = [Datastore(name=name) for name in env.names]

## Collect transitions

Each step merges the input we sent (`{"action": …}`) with the environment's output (`observation`, `reward`, `done`, `info_metadata_q_star`, …) into a single dict and appends it to the matching store.

**Oracle ramp:** at step `t`, `oracle_prob = t / (TOTAL_STEPS − 1)`. Each slot independently samples: with probability `oracle_prob` it picks `argmax(info_metadata_q_star)` (the optimal action); otherwise it picks a random action. This means the first steps explore freely while later steps demonstrate expert behaviour.

**Initial reset frame:** the very first `env.step` triggers the environment reset; the env ignores the action and returns the initial observation. We store this frame so every datastore starts from timestep 0 and is time-aligned.

In [ ]:
def pick_inputs(outputs, env, oracle_prob: float) -> list[dict]:
    random_inputs = env.sample_random_inputs()
    inputs = []
    for i, out in enumerate(outputs):
        if torch.rand(1).item() >= oracle_prob:
            inputs.append(random_inputs[i])
        else:
            action = out.get("info_q_star").argmax()
            inputs.append({"action": torch.tensor(action, dtype=torch.int64)})
    return inputs

outputs = None
env.tracker.clear()
for t in range(TOTAL_STEPS):
    oracle_prob = t / (TOTAL_STEPS - 1)
    if outputs is None:
        inputs = env.sample_random_inputs()
    else:
        inputs = pick_inputs(outputs, env, oracle_prob)
    outputs = env.step(inputs)
    for store, inp, out in zip(stores, inputs, outputs):
        store.append({**inp, **out})

env.close()

for name, rewards, lengths in zip(env.names, env.tracker.episode_cum_rewards, env.tracker.episode_lengths):
    n = len(rewards)
    mean_r = torch.tensor(rewards).mean().item() if n else float("nan")
    mean_l = torch.tensor(lengths).mean().item() if n else float("nan")
    print(f"{name}: {n} episodes | mean reward {mean_r:.3f} | mean length {mean_l:.1f}")

[{'action': tensor(2)}, {'action': tensor(2)}, {'action': tensor(1)}, {'action': tensor(3)}]
[{'action': tensor(1)}, {'action': tensor(1)}, {'action': tensor(0)}, {'action': tensor(3)}]
[{'action': tensor(1)}, {'action': tensor(3)}, {'action': tensor(3)}, {'action': tensor(1)}]
[{'action': tensor(1)}, {'action': tensor(2)}, {'action': tensor(2)}, {'action': tensor(2)}]
[{'action': tensor(0)}, {'action': tensor(2)}, {'action': tensor(0)}, {'action': tensor(2)}]
[{'action': tensor(2)}, {'action': tensor(0)}, {'action': tensor(0)}, {'action': tensor(3)}]
[{'action': tensor(1)}, {'action': tensor(1)}, {'action': tensor(0)}, {'action': tensor(0)}]
[{'action': tensor(3)}, {'action': tensor(3)}, {'action': tensor(0)}, {'action': tensor(1)}]
[{'action': tensor(3)}, {'action': tensor(3)}, {'action': tensor(3)}, {'action': tensor(2)}]
[{'action': tensor(0)}, {'action': tensor(1)}, {'action': tensor(3)}, {'action': tensor(0)}]
[{'action': tensor(3)}, {'action': tensor(1)}, {'action': tensor(0)}, 

## Push to the Hub

Upload all four stores to your Hugging Face account as a single dataset repo. Each store is saved as a separate named config, which is how `load_stores_from_hub` retrieves them by name in the training notebook.

In [28]:
url = push_stores_to_hub(stores, repo_id=DATASET_ID, split="train", private=False)
print(f"Pushed to {url}")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed to micahr234/mouse-example-dataset (proc_frozenlake_0/train: 1500, proc_frozenlake_1/train: 1500, proc_frozenlake_2/train: 1500, proc_frozenlake_3/train: 1500 steps)
Pushed to https://huggingface.co/datasets/micahr234/mouse-example-dataset
